# ML Exploration
Use this notebook to experiment with models before moving code to `model/train.py`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

## 1. Load processed data

In [2]:
TICKER = 'AAPL'
df = pd.read_csv(f'../data/processed/{TICKER}.csv', parse_dates=['Date'])

# Features: all engineered columns — raw prices excluded (non-stationary)
FEATURES = [
    'Return', 'MA5', 'MA20', 'Volume_Change',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
    'Return_Lag1', 'Return_Lag2'
]

X = df[FEATURES]
y = df['Target']

print(f"Dataset: {len(df)} rows | {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Features: {FEATURES}")
print(f"\nTarget balance:\n{y.value_counts()}")

Dataset: 1217 rows | 2020-05-05 → 2025-03-07
Features: ['Return', 'MA5', 'MA20', 'Volume_Change', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'Return_Lag1', 'Return_Lag2']

Target balance:
Target
1    649
0    568
Name: count, dtype: int64


## 2. Train / test split

In [ ]:

# Time-series split: 80% train, 20% test — ordered by time, NEVER shuffled.
# Shuffling would leak future prices into training, making results look better than they are.
split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} rows  {df['Date'].iloc[0].date()} → {df['Date'].iloc[split_idx-1].date()}")
print(f"Test:  {len(X_test)} rows  {df['Date'].iloc[split_idx].date()} → {df['Date'].iloc[-1].date()}")
print(f"\nClass balance (test set):\n{y_test.value_counts()}")


## 3. Model selection & evaluation

In [ ]:

# Model 1: Logistic Regression — simplest reasonable baseline.
# StandardScaler is required: LR is sensitive to feature magnitudes.
# RSI ranges 0–100, MACD is in tiny decimals — without scaling, LR struggles.
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['Down (0)', 'Up (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_pipe.predict_proba(X_test)[:, 1]):.3f}")


In [ ]:

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=['Down', 'Up'], ax=ax
)
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()


## 4. Model 2: Random Forest

In [ ]:

# Model 2: Random Forest — captures non-linear patterns, no scaling needed.
# n_jobs=-1 uses all CPU cores for faster training.
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),  # kept for pipeline consistency; RF doesn't need it
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Down (0)', 'Up (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_pipe.predict_proba(X_test)[:, 1]):.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=['Down', 'Up'], ax=axes[0]
)
axes[0].set_title('Random Forest — Confusion Matrix')

importances = rf_pipe.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1])
axes[1].set_title('Random Forest — Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()


## 5. Export best model

In [ ]:

# Export the best-performing model to model/trained/.
# joblib is preferred over pickle for sklearn objects containing numpy arrays.
# The web app will load this exact file to make real-time predictions.

best_model = rf_pipe  # swap to lr_pipe if LR scores higher
model_path = f'../model/trained/model_{TICKER}.pkl'

os.makedirs('../model/trained', exist_ok=True)
joblib.dump(best_model, model_path)
print(f"Saved → {model_path}")
print(f"Features used (in order): {FEATURES}")
